# FrameFill – GOES-19 C13 interpolation (Kaggle-safe)
Run cells top to bottom. **Set Accelerator (GPU T4) and Internet ON *before* running** – changing them restarts the session and wipes `/kaggle/working`.
Every stage skips work whose output file already exists.

In [ ]:
# CELL 1 - environment
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
import sys, subprocess, importlib
for mod, pipname in [("s3fs","s3fs"),("h5netcdf","h5netcdf"),("xarray","xarray"),("cv2","opencv-python-headless"),("skimage","scikit-image")]:
    try: importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable,"-m","pip","install","-q",pipname])
import torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
os.makedirs("/kaggle/working", exist_ok=True)

In [ ]:
# CELL 2 - download GOES-19 C13 crops (skipped if already saved)
import s3fs, xarray as xr, numpy as np, re, os
from concurrent.futures import ThreadPoolExecutor
fs = s3fs.S3FileSystem(anon=True)
Y0, X0, S = 2400, 2400, 512

def parse_time(key):
    m = re.search(r"_s(\d{4})(\d{3})(\d{2})(\d{2})", key)
    y, d, hh, mm = map(int, m.groups())
    return np.datetime64(f"{y}-01-01") + np.timedelta64(d-1,"D") + np.timedelta64(hh*60+mm,"m")

def load_crop(key):
    with fs.open(key) as f:
        ds = xr.open_dataset(f, engine="h5netcdf", mask_and_scale=True)
        arr = ds["CMI"].isel(y=slice(Y0,Y0+S), x=slice(X0,X0+S)).values.astype("float32")
        ds.close()
        return arr

def safe(k):
    try: return load_crop(k), parse_time(k)
    except Exception as e:
        print("skip", k.split("/")[-1], repr(e)[:100]); return None

def grab(day, name):
    out_path = f"/kaggle/working/{name}.npz"
    if os.path.exists(out_path):
        print(name, "exists, skipping"); return
    files = sorted(f for h in fs.ls(day) for f in fs.ls(h) if "C13" in f and f.endswith(".nc"))
    print(day, len(files), "files")
    with ThreadPoolExecutor(4) as ex:
        out = [r for r in ex.map(safe, files) if r is not None]
    fr = np.stack([r[0] for r in out]); tm = np.array([r[1] for r in out])
    np.savez_compressed(out_path, frames=fr, times=tm)
    print(name, fr.shape, np.nanmin(fr), np.nanmax(fr))

grab("noaa-goes19/ABI-L2-CMIPF/2026/295", "crops_dayA")
grab("noaa-goes19/ABI-L2-CMIPF/2026/292", "crops_dayB")

In [ ]:
# CELL 3 - triplets, metrics, baselines B0/B1
import cv2, pandas as pd
from skimage.metrics import structural_similarity as ssim
from scipy.ndimage import uniform_filter
LO, HI = 180.0, 330.0

def build_triplets(frames, times):
    t = []
    for i in range(len(times)-2):
        if (times[i+2]-times[i]) == np.timedelta64(20,"m") and max(np.isnan(frames[j]).mean() for j in (i,i+1,i+2)) < 0.2:
            t.append((i, i+1, i+2))
    return t

d = np.load("/kaggle/working/crops_dayA.npz"); frames, times = d["frames"], d["times"]
triplets = build_triplets(frames, times)
dB = np.load("/kaggle/working/crops_dayB.npz"); framesB, timesB = dB["frames"], dB["times"]
tripletsB = build_triplets(framesB, timesB)
print("Day A triplets:", len(triplets), "| Day B triplets:", len(tripletsB))
np.save("/kaggle/working/triplets_A.npy", np.array(triplets))
np.save("/kaggle/working/triplets_B.npy", np.array(tripletsB))

def fill(x): return np.where(np.isnan(x), np.nanmean(x), x).astype("float32")
def to8(x):  return (np.clip((fill(x)-LO)/(HI-LO),0,1)*255).astype("uint8")
def b0(f0, f1): return 0.5*f0 + 0.5*f1

dis = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM)
def b1(f0, f1):
    a, b = fill(f0), fill(f1)
    fl01 = dis.calc(to8(f0), to8(f1), None); fl10 = dis.calc(to8(f1), to8(f0), None)
    h, w = a.shape
    gx, gy = np.meshgrid(np.arange(w, dtype="float32"), np.arange(h, dtype="float32"))
    w0 = cv2.remap(a, gx-0.5*fl01[...,0], gy-0.5*fl01[...,1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    w1 = cv2.remap(b, gx-0.5*fl10[...,0], gy-0.5*fl10[...,1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    out = 0.5*w0 + 0.5*w1
    out[np.isnan(f0) | np.isnan(f1)] = np.nan
    return out

def metrics(P, G, f0, f1):
    V = np.isfinite(P) & np.isfinite(G) & np.isfinite(f0) & np.isfinite(f1)
    D = V & (uniform_filter(np.nan_to_num(np.abs(f0-f1)), 3) > 3.0)
    def mr(m):
        if m.sum() < 50: return np.nan, np.nan
        e = (P-G)[m]; return np.abs(e).mean(), np.sqrt((e**2).mean())
    mae, rmse = mr(V); mae_d, rmse_d = mr(D)
    n = lambda x: np.clip((np.nan_to_num(x, nan=LO)-LO)/(HI-LO),0,1)
    return dict(MAE=mae, RMSE=rmse, PSNR=10*np.log10(150**2/max(rmse**2,1e-9)),
                SSIM=ssim(n(P), n(G), data_range=1.0, win_size=7),
                MAE_dyn=mae_d, RMSE_dyn=rmse_d, dyn_frac=D.mean())

def eval_engine(engine, fr, trips, name):
    rows = [metrics(engine(fr[a], fr[b]), fr[m], fr[a], fr[b]) for a,m,b in trips]
    print(name, len(rows)); return pd.DataFrame(rows)

def eval_file(path, fr, trips):
    P = np.load(path)["preds"]
    return pd.DataFrame([metrics(P[k], fr[m], fr[a], fr[b]) for k,(a,m,b) in enumerate(trips[:len(P)])])

base = {"B0": eval_engine(b0, frames, triplets, "B0"),
        "B1": eval_engine(b1, frames, triplets, "B1")}
tab0 = pd.DataFrame({k: v.mean() for k, v in base.items()}).T.round(3)
print(tab0); tab0.to_csv("/kaggle/working/dayA_baselines.csv")

In [ ]:
# CELL 4 - prepare a writable RIFE package (copy code + weights, stub training-only loss)
import shutil, os
SRC_REPO = "/kaggle/input/datasets/arpitdeosthale/practical-rife-code/Practical-RIFE"
SRC_WTS  = "/kaggle/input/datasets/arpitdeosthale/rife-weights/RIFEv4.22"
PKG = "/kaggle/working/rife_pkg"
assert os.path.isdir(SRC_REPO + "/model"), "RIFE code dataset not attached"
assert os.path.isfile(SRC_WTS + "/train_log/flownet.pkl"), "RIFE weights dataset not attached"
ign = shutil.ignore_patterns("__pycache__", "*.pyc")
shutil.copytree(SRC_REPO + "/model", PKG + "/model", ignore=ign, dirs_exist_ok=True)
shutil.copytree(SRC_WTS + "/train_log", PKG + "/train_log", ignore=ign, dirs_exist_ok=True)
for dname in ("model", "train_log"):
    open(f"{PKG}/{dname}/__init__.py", "a").close()
print(sorted(os.listdir(PKG + "/model")), sorted(os.listdir(PKG + "/train_log")))

In [ ]:
%%writefile /kaggle/working/rife_run.py
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
import sys, argparse, types, traceback
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

p = argparse.ArgumentParser()
p.add_argument("--data", default="/kaggle/working/crops_dayA.npz")
p.add_argument("--trips", default="/kaggle/working/triplets_A.npy")
p.add_argument("--out", required=True)
p.add_argument("--mode", default="fixed")      # fixed | naive
p.add_argument("--scale", type=float, default=1.0)
p.add_argument("--invert", type=int, default=0)
p.add_argument("--limit", type=int, default=0)
a = p.parse_args()
PKG = "/kaggle/working/rife_pkg"

try:
    torch.set_num_threads(4)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sys.path.insert(0, PKG)

    import model.warplayer
    try:
        import model.loss
    except Exception as e:
        print("stubbing model.loss:", repr(e), flush=True)
        stub = types.ModuleType("model.loss")
        class _Noop(nn.Module):
            def __init__(self, *args, **kw): super().__init__()
            def forward(self, *args, **kw): return torch.zeros(1)
        for n in ("EPE", "SOBEL", "Ternary", "VGGPerceptualLoss", "MeanShift"):
            setattr(stub, n, _Noop)
        sys.modules["model.loss"] = stub
        model.loss = stub

    from train_log.RIFE_HDv3 import Model
    net = Model()
    net.load_model(PKG + "/train_log", -1)
    net.eval(); net.device()
    print("model ready on", dev, flush=True)

    LO, HI = 180.0, 330.0
    frames = np.load(a.data)["frames"]; trips = np.load(a.trips)
    if a.limit: trips = trips[:a.limit]

    def fill(x): return np.where(np.isnan(x), np.nanmean(x), x).astype("float32")
    PAD = max(128, int(128 / a.scale))
    def prep(x, lo, hi):
        x = np.clip((fill(x) - lo) / (hi - lo), 0, 1)
        t = torch.from_numpy(x)[None, None].repeat(1, 3, 1, 1).float().to(dev)
        h, w = t.shape[-2:]
        ph, pw = ((h - 1) // PAD + 1) * PAD, ((w - 1) // PAD + 1) * PAD
        return F.pad(t, (0, pw - w, 0, ph - h), mode="replicate"), h, w

    preds = []
    with torch.no_grad():
        for i, (ia, im, ib) in enumerate(trips):
            f0, f1 = frames[ia], frames[ib]
            if a.mode == "fixed":
                lo0 = lo1 = LO; hi0 = hi1 = HI
            else:
                lo0, hi0 = np.nanmin(f0), np.nanmax(f0); lo1, hi1 = np.nanmin(f1), np.nanmax(f1)
            x0, h, w = prep(f0, lo0, hi0); x1, _, _ = prep(f1, lo1, hi1)
            if a.invert: x0, x1 = 1 - x0, 1 - x1
            out = net.inference(x0, x1, 0.5, a.scale)
            if a.invert: out = 1 - out
            y = out.mean(1)[0, :h, :w].clamp(0, 1).cpu().numpy()
            lo, hi = (lo0 + lo1) / 2, (hi0 + hi1) / 2
            bt = y * (hi - lo) + lo
            bt = np.clip(bt, min(np.nanmin(f0), np.nanmin(f1)) - 5, max(np.nanmax(f0), np.nanmax(f1)) + 5)
            bt[np.isnan(f0) | np.isnan(f1)] = np.nan
            preds.append(bt.astype("float32"))
            if i % 20 == 0: print("frame", i, flush=True)
    np.savez_compressed(a.out, preds=np.stack(preds), trips=trips)
    print("done", len(preds), flush=True)
except Exception:
    traceback.print_exc()
    sys.exit(1)

In [ ]:
# CELL 6 - runner that cannot kill the notebook
import subprocess, sys
def run_rife(*args):
    r = subprocess.run([sys.executable, "-X", "faulthandler", "/kaggle/working/rife_run.py", *args],
                       capture_output=True, text=True)
    print("\n".join((r.stdout + r.stderr).splitlines()[-40:]))
    print("exit code:", r.returncode)
    return r.returncode == 0

ok = run_rife("--out", "/kaggle/working/p_test.npz", "--limit", "3")   # smoke test

In [ ]:
# CELL 7 - full RIFE runs (only if smoke test passed; skips finished outputs)
import os
if not ok:
    print("Smoke test failed - fix that first (see traceback above).")
else:
    if not os.path.exists("/kaggle/working/A_B3.npz"):
        run_rife("--out", "/kaggle/working/A_B3.npz", "--mode", "fixed")
    if not os.path.exists("/kaggle/working/A_B2.npz"):
        run_rife("--out", "/kaggle/working/A_B2.npz", "--mode", "naive")

In [ ]:
# CELL 8 - results table
res = dict(base)
for label, path in [("B2 naive","/kaggle/working/A_B2.npz"), ("B3 fixed","/kaggle/working/A_B3.npz")]:
    if os.path.exists(path):
        res[label] = eval_file(path, frames, triplets)
tab = pd.DataFrame({k: v.mean() for k, v in res.items()}).T.round(3)
print(tab); tab.to_csv("/kaggle/working/dayA_results.csv")

In [ ]:
import os, glob, shutil
os.makedirs("/kaggle/working", exist_ok=True)
names = ["crops_dayA.npz", "crops_dayB.npz", "triplets_A.npy", "triplets_B.npy",
         "A_B3.npz", "A_B2.npz", "dayA_baselines.csv", "dayA_results.csv"]
for n in names:
    dst = f"/kaggle/working/{n}"
    if os.path.exists(dst):
        continue
    hits = glob.glob(f"/kaggle/input/**/{n}", recursive=True)
    if hits:
        shutil.copy(hits[0], dst)
        print("restored", n, "<-", hits[0])
    else:
        print("NOT FOUND", n)

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import numpy as np, pandas as pd, cv2
from skimage.metrics import structural_similarity as ssim
from scipy.ndimage import uniform_filter

LO, HI = 180.0, 330.0
W = "/kaggle/working"
frames = np.load(f"{W}/crops_dayA.npz")["frames"]
triplets = np.load(f"{W}/triplets_A.npy")

def fill(x): return np.where(np.isnan(x), np.nanmean(x), x).astype("float32")
def to8(x):  return (np.clip((fill(x)-LO)/(HI-LO),0,1)*255).astype("uint8")
def b0(f0, f1): return 0.5*f0 + 0.5*f1

dis = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM)
def b1(f0, f1):
    a, b = fill(f0), fill(f1)
    fl01 = dis.calc(to8(f0), to8(f1), None); fl10 = dis.calc(to8(f1), to8(f0), None)
    h, w = a.shape
    gx, gy = np.meshgrid(np.arange(w, dtype="float32"), np.arange(h, dtype="float32"))
    w0 = cv2.remap(a, gx-0.5*fl01[...,0], gy-0.5*fl01[...,1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    w1 = cv2.remap(b, gx-0.5*fl10[...,0], gy-0.5*fl10[...,1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    out = 0.5*w0 + 0.5*w1
    out[np.isnan(f0) | np.isnan(f1)] = np.nan
    return out

def metrics(P, G, f0, f1):
    V = np.isfinite(P) & np.isfinite(G) & np.isfinite(f0) & np.isfinite(f1)
    D = V & (uniform_filter(np.nan_to_num(np.abs(f0-f1)), 3) > 3.0)
    def mr(m):
        if m.sum() < 50: return np.nan, np.nan
        e = (P-G)[m]; return np.abs(e).mean(), np.sqrt((e**2).mean())
    mae, rmse = mr(V); mae_d, rmse_d = mr(D)
    n = lambda x: np.clip((np.nan_to_num(x, nan=LO)-LO)/(HI-LO),0,1)
    return dict(MAE=mae, RMSE=rmse, PSNR=10*np.log10(150**2/max(rmse**2,1e-9)),
                SSIM=ssim(n(P), n(G), data_range=1.0, win_size=7),
                MAE_dyn=mae_d, RMSE_dyn=rmse_d, dyn_frac=D.mean())

def eval_engine(engine):
    return pd.DataFrame([metrics(engine(frames[a], frames[b]), frames[m], frames[a], frames[b]) for a,m,b in triplets])
def eval_file(path):
    P = np.load(path)["preds"]
    return pd.DataFrame([metrics(P[k], frames[m], frames[a], frames[b]) for k,(a,m,b) in enumerate(triplets[:len(P)])])

res = {"B0": eval_engine(b0), "B1": eval_engine(b1),
       "B2 naive": eval_file(f"{W}/A_B2.npz"), "B3 fixed": eval_file(f"{W}/A_B3.npz")}
tab = pd.DataFrame({k: v.mean() for k, v in res.items()}).T.round(3)
print(tab)
tab.to_csv(f"{W}/dayA_results.csv")

# paired bootstrap over triplets: positive diff means the first engine has higher error
def boot(a, b, col="RMSE_dyn", n=1000, seed=42):
    d = (res[a][col] - res[b][col]).dropna().values
    rng = np.random.default_rng(seed)
    means = [rng.choice(d, len(d), replace=True).mean() for _ in range(n)]
    lo, hi = np.percentile(means, [2.5, 97.5])
    print(f"{a} - {b} on {col}: mean diff {d.mean():+.3f} K, 95% CI [{lo:+.3f}, {hi:+.3f}], N={len(d)}")

boot("B3 fixed", "B1")      # does RIFE differ from classical flow?
boot("B2 naive", "B3 fixed") # does fixed-range normalisation help?
boot("B0", "B1")             # sanity: flow vs blend

In [ ]:
import os, shutil, subprocess, sys
W = "/kaggle/working"
SRC = "/kaggle/input/notebooks/arpitdeosthale/be-project"

# restore the runner and the RIFE package from your saved output
shutil.copytree(f"{SRC}/rife_pkg", f"{W}/rife_pkg", dirs_exist_ok=True)
shutil.copy(f"{SRC}/rife_run.py", f"{W}/rife_run.py")
print("package:", sorted(os.listdir(f"{W}/rife_pkg")),
      "| weights:", os.path.exists(f"{W}/rife_pkg/train_log/flownet.pkl"))

def run_rife(*args):
    r = subprocess.run([sys.executable, "-X", "faulthandler", f"{W}/rife_run.py", *args],
                       capture_output=True, text=True)
    print("\n".join((r.stdout + r.stderr).splitlines()[-8:]))
    return r.returncode == 0

dayB = ["--data", f"{W}/crops_dayB.npz", "--trips", f"{W}/triplets_B.npy"]
jobs = [("A_B3_s05", ["--scale", "0.5"]),
        ("A_B3_inv", ["--invert", "1"]),
        ("B_B3",     dayB),
        ("B_B2",     dayB + ["--mode", "naive"])]
for name, extra in jobs:
    out = f"{W}/{name}.npz"
    if os.path.exists(out):
        print(name, "exists, skipping"); continue
    print(name, "->", "ok" if run_rife("--out", out, *extra) else "FAILED")

In [ ]:
framesB = np.load(f"{W}/crops_dayB.npz")["frames"]
tripsB  = np.load(f"{W}/triplets_B.npy")

def eval_eng(engine, fr, trips):
    return pd.DataFrame([metrics(engine(fr[a], fr[b]), fr[m], fr[a], fr[b]) for a,m,b in trips])
def eval_f(path, fr, trips):
    P = np.load(path)["preds"]
    return pd.DataFrame([metrics(P[k], fr[m], fr[a], fr[b]) for k,(a,m,b) in enumerate(trips[:len(P)])])
def boot2(R, a, b, col="RMSE_dyn", n=1000, seed=42):
    d = (R[a][col] - R[b][col]).dropna().values
    rng = np.random.default_rng(seed)
    m = [rng.choice(d, len(d), replace=True).mean() for _ in range(n)]
    lo, hi = np.percentile(m, [2.5, 97.5])
    print(f"  {a} - {b}: {d.mean():+.3f} K, 95% CI [{lo:+.3f}, {hi:+.3f}], N={len(d)}")

# ---- Day B hold-out ----
resB = {"B0": eval_eng(b0, framesB, tripsB), "B1": eval_eng(b1, framesB, tripsB),
        "B2 naive": eval_f(f"{W}/B_B2.npz", framesB, tripsB),
        "B3 fixed": eval_f(f"{W}/B_B3.npz", framesB, tripsB)}
tabB = pd.DataFrame({k: v.mean() for k, v in resB.items()}).T.round(3)
print("=== Day B (held-out) ===\n", tabB)
boot2(resB, "B3 fixed", "B1"); boot2(resB, "B0", "B1"); boot2(resB, "B2 naive", "B3 fixed")
tabB.to_csv(f"{W}/dayB_results.csv")

# ---- RIFE ablations on Day A ----
abl = {"B1": res["B1"], "B3 fixed": res["B3 fixed"],
       "B3 scale0.5": eval_f(f"{W}/A_B3_s05.npz", frames, triplets),
       "B3 invert":   eval_f(f"{W}/A_B3_inv.npz", frames, triplets)}
tabA = pd.DataFrame({k: v.mean() for k, v in abl.items()}).T.round(3)
print("\n=== Day A ablations ===\n", tabA)
boot2(abl, "B3 scale0.5", "B3 fixed"); boot2(abl, "B3 invert", "B3 fixed"); boot2(abl, "B3 scale0.5", "B1")
tabA.to_csv(f"{W}/dayA_ablations.csv")

# ---- error vs displacement (Day A): does RIFE win on fast motion? ----
def mean_disp(f0, f1):
    fl = dis.calc(to8(f0), to8(f1), None)
    return float(np.hypot(fl[...,0], fl[...,1]).mean())
disp = np.array([mean_disp(frames[a], frames[b]) for a,m,b in triplets])
bins = pd.qcut(disp, 3, labels=["slow", "medium", "fast"])
rows = {k: pd.Series(abl[k]["RMSE_dyn"].values[:len(disp)]).groupby(bins).mean()
        for k in ["B1", "B3 fixed", "B3 scale0.5"]}
print("\n=== RMSE_dyn (K) by motion tercile ===")
print(pd.DataFrame(rows).round(3))
print("mean displacement per tercile (px / 20 min):",
      pd.Series(disp).groupby(bins).mean().round(2).to_dict())

In [ ]:
import os, subprocess, sys
W = "/kaggle/working"
assert os.path.exists(f"{W}/rife_run.py") and os.path.isdir(f"{W}/rife_pkg"), "rerun the restore cell first"

def run_rife(*args):
    r = subprocess.run([sys.executable, "-X", "faulthandler", f"{W}/rife_run.py", *args],
                       capture_output=True, text=True)
    print("\n".join((r.stdout + r.stderr).splitlines()[-4:]))
    return r.returncode == 0

dayB = ["--data", f"{W}/crops_dayB.npz", "--trips", f"{W}/triplets_B.npy"]
jobs = []
for s in ("1.5", "2.0"):
    tag = s.replace(".", "")
    jobs.append((f"A_B3_s{tag}", ["--scale", s]))
    jobs.append((f"B_B3_s{tag}", dayB + ["--scale", s]))

for name, extra in jobs:
    out = f"{W}/{name}.npz"
    if os.path.exists(out):
        print(name, "exists, skipping"); continue
    print(name, "->", "ok" if run_rife("--out", out, *extra) else "FAILED")

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import numpy as np, pandas as pd, cv2
from skimage.metrics import structural_similarity as ssim
from scipy.ndimage import uniform_filter

W = "/kaggle/working"
LO, HI = 180.0, 330.0
data = {"A": (np.load(f"{W}/crops_dayA.npz")["frames"], np.load(f"{W}/triplets_A.npy")),
        "B": (np.load(f"{W}/crops_dayB.npz")["frames"], np.load(f"{W}/triplets_B.npy"))}

def fill(x): return np.where(np.isnan(x), np.nanmean(x), x).astype("float32")
def to8(x):  return (np.clip((fill(x)-LO)/(HI-LO),0,1)*255).astype("uint8")
dis = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM)
def b1(f0, f1):
    a, b = fill(f0), fill(f1)
    fl01 = dis.calc(to8(f0), to8(f1), None); fl10 = dis.calc(to8(f1), to8(f0), None)
    h, w = a.shape
    gx, gy = np.meshgrid(np.arange(w, dtype="float32"), np.arange(h, dtype="float32"))
    w0 = cv2.remap(a, gx-0.5*fl01[...,0], gy-0.5*fl01[...,1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    w1 = cv2.remap(b, gx-0.5*fl10[...,0], gy-0.5*fl10[...,1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    out = 0.5*w0 + 0.5*w1
    out[np.isnan(f0) | np.isnan(f1)] = np.nan
    return out

def metrics(P, G, f0, f1):
    V = np.isfinite(P) & np.isfinite(G) & np.isfinite(f0) & np.isfinite(f1)
    D = V & (uniform_filter(np.nan_to_num(np.abs(f0-f1)), 3) > 3.0)
    def mr(m):
        if m.sum() < 50: return np.nan, np.nan
        e = (P-G)[m]; return np.abs(e).mean(), np.sqrt((e**2).mean())
    mae, rmse = mr(V); mae_d, rmse_d = mr(D)
    n = lambda x: np.clip((np.nan_to_num(x, nan=LO)-LO)/(HI-LO),0,1)
    return dict(MAE=mae, RMSE=rmse, PSNR=10*np.log10(150**2/max(rmse**2,1e-9)),
                SSIM=ssim(n(P), n(G), data_range=1.0, win_size=7),
                MAE_dyn=mae_d, RMSE_dyn=rmse_d, dyn_frac=D.mean())

def score(P, fr, trips):
    return pd.DataFrame([metrics(P[k], fr[m], fr[a], fr[b]) for k,(a,m,b) in enumerate(trips[:len(P)])])

def boot2(R, a, b, col="RMSE_dyn", n=1000, seed=42):
    d = (R[a][col] - R[b][col]).dropna().values
    rng = np.random.default_rng(seed)
    m = [rng.choice(d, len(d), replace=True).mean() for _ in range(n)]
    lo, hi = np.percentile(m, [2.5, 97.5])
    print(f"  {a} - {b}: {d.mean():+.3f} K, 95% CI [{lo:+.3f}, {hi:+.3f}], N={len(d)}")

def rife_file(day, tag):
    name = {"A": {"1.0": "A_B3", "1.5": "A_B3_s15", "2.0": "A_B3_s20"},
            "B": {"1.0": "B_B3", "1.5": "B_B3_s15", "2.0": "B_B3_s20"}}[day][tag]
    return np.load(f"{W}/{name}.npz")["preds"]

for day in ("A", "B"):
    fr, trips = data[day]
    P1 = np.stack([b1(fr[a], fr[b]) for a,m,b in trips])
    R = {"B1": score(P1, fr, trips)}
    preds = {t: rife_file(day, t) for t in ("1.0", "1.5", "2.0")}
    for t, P in preds.items():
        R[f"B3 s{t}"] = score(P, fr, trips)
    for t in ("1.0", "2.0"):
        R[f"blend B1+B3 s{t}"] = score(0.5*P1 + 0.5*preds[t], fr, trips)
    tab = pd.DataFrame({k: v.mean() for k, v in R.items()}).T.round(3)
    print(f"\n=== Day {day} ===\n", tab[["MAE","RMSE","SSIM","MAE_dyn","RMSE_dyn"]])
    tab.to_csv(f"{W}/day{day}_scale_blend.csv")
    best_rife = min(("B3 s1.0","B3 s1.5","B3 s2.0"), key=lambda k: R[k]["RMSE_dyn"].mean())
    best_blend = min(("blend B1+B3 s1.0","blend B1+B3 s2.0"), key=lambda k: R[k]["RMSE_dyn"].mean())
    print(f"best RIFE: {best_rife} | best blend: {best_blend}")
    boot2(R, best_rife, "B1"); boot2(R, best_blend, "B1")

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import numpy as np, pandas as pd, cv2

W = "/kaggle/working"
LO, HI = 180.0, 330.0
data = {"A": (np.load(f"{W}/crops_dayA.npz")["frames"], np.load(f"{W}/triplets_A.npy")),
        "B": (np.load(f"{W}/crops_dayB.npz")["frames"], np.load(f"{W}/triplets_B.npy"))}

def fill(x): return np.where(np.isnan(x), np.nanmean(x), x).astype("float32")
def to8(x):  return (np.clip((fill(x)-LO)/(HI-LO),0,1)*255).astype("uint8")
dis = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM)
def b0(f0, f1): return 0.5*f0 + 0.5*f1
def b1(f0, f1):
    a, b = fill(f0), fill(f1)
    fl01 = dis.calc(to8(f0), to8(f1), None); fl10 = dis.calc(to8(f1), to8(f0), None)
    h, w = a.shape
    gx, gy = np.meshgrid(np.arange(w, dtype="float32"), np.arange(h, dtype="float32"))
    w0 = cv2.remap(a, gx-0.5*fl01[...,0], gy-0.5*fl01[...,1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    w1 = cv2.remap(b, gx-0.5*fl10[...,0], gy-0.5*fl10[...,1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    out = 0.5*w0 + 0.5*w1
    out[np.isnan(f0) | np.isnan(f1)] = np.nan
    return out

# ---- sharpness metrics ----
def grad_mag(x):
    gx = cv2.Sobel(x, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(x, cv2.CV_32F, 0, 1, ksize=3)
    return np.hypot(gx, gy)

_hann = None
def hf_power(x):
    global _hann
    n = x.shape[0]
    if _hann is None or _hann.shape[0] != n:
        _hann = np.outer(np.hanning(n), np.hanning(n)).astype("float32")
    x = (x - x.mean()) * _hann
    F = np.fft.fftshift(np.fft.fft2(x))
    P = np.abs(F)**2
    yy, xx = np.indices(P.shape); c = n//2
    r = np.hypot(yy-c, xx-c) / (n/2)
    return P[r >= 0.75].sum()          # top 25% of wavenumbers

def sharp_metrics(P, G):
    V = np.isfinite(P) & np.isfinite(G)
    ref = np.nanmean(G)
    Pf = np.where(V, P, ref).astype("float32"); Gf = np.where(V, G, ref).astype("float32")
    gc = np.corrcoef(grad_mag(Pf).ravel(), grad_mag(Gf).ravel())[0, 1]
    return dict(grad_corr=gc,
                grad_mag_ratio=grad_mag(Pf)[V].mean() / grad_mag(Gf)[V].mean(),
                hf_ratio=hf_power(Pf) / hf_power(Gf),
                bias_K=(P - G)[V].mean())

def rife(day, tag):
    return np.load(f"{W}/{ {'A': 'A_B3_s20', 'B': 'B_B3_s20'}[day] }.npz")["preds"]

for day in ("A", "B"):
    fr, trips = data[day]
    G  = np.stack([fr[m] for a, m, b in trips])
    P0 = np.stack([b0(fr[a], fr[b]) for a, m, b in trips])
    P1 = np.stack([b1(fr[a], fr[b]) for a, m, b in trips])
    P3 = rife(day, "s20")[:len(trips)]
    PB = 0.5*P1 + 0.5*P3
    engines = {"B0 average": P0, "B1 DIS flow": P1, "B3 RIFE s2.0": P3, "Blend B1+B3": PB}
    rows = {k: pd.DataFrame([sharp_metrics(P[i], G[i]) for i in range(len(G))]).mean()
            for k, P in engines.items()}
    print(f"\n=== Day {day}: sharpness vs observed frame (1.0 = as sharp as the truth) ===")
    print(pd.DataFrame(rows).T.round(3))

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import numpy as np, xarray as xr, cv2

W = "/kaggle/working"
LO, HI = 180.0, 330.0
d = np.load(f"{W}/crops_dayA.npz"); frames, times = d["frames"], d["times"]
trips = np.load(f"{W}/triplets_A.npy")
P3 = np.load(f"{W}/A_B3_s20.npz")["preds"]          # RIFE scale 2.0, one prediction per triplet
row = {int(a): k for k, (a, m, b) in enumerate(trips)}   # first-frame index -> prediction row

def fill(x): return np.where(np.isnan(x), np.nanmean(x), x).astype("float32")
def to8(x):  return (np.clip((fill(x)-LO)/(HI-LO),0,1)*255).astype("uint8")
dis = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM)
def b1(f0, f1):
    a, b = fill(f0), fill(f1)
    fl01 = dis.calc(to8(f0), to8(f1), None); fl10 = dis.calc(to8(f1), to8(f0), None)
    h, w = a.shape
    gx, gy = np.meshgrid(np.arange(w, dtype="float32"), np.arange(h, dtype="float32"))
    w0 = cv2.remap(a, gx-0.5*fl01[...,0], gy-0.5*fl01[...,1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    w1 = cv2.remap(b, gx-0.5*fl10[...,0], gy-0.5*fl10[...,1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    out = 0.5*w0 + 0.5*w1
    out[np.isnan(f0) | np.isnan(f1)] = np.nan
    return out

# observed frames 20 min apart: pairs (i, i+2), (i+2, i+4), ... each needs a triplet starting at its first frame
def run_from(s, max_pairs=36):
    pairs, cur = [], s
    while cur in row and len(pairs) < max_pairs:
        pairs.append(cur); cur += 2
    return pairs
best = max((run_from(s) for s in sorted(row)), key=len)
print("20-min pairs in the sequence:", len(best), "| first observed frame index:", best[0])

seq, ftype, ifrac, tt = [], [], [], []
for i in best:
    seq.append(frames[i]); ftype.append(0); ifrac.append(0.0); tt.append(times[i])
    mid = 0.5 * b1(frames[i], frames[i+2]) + 0.5 * P3[row[i]]
    seq.append(mid.astype("float32")); ftype.append(1); ifrac.append(0.5)
    tt.append(times[i] + np.timedelta64(10, "m"))
last = best[-1] + 2                                   # close the sequence with the final observed frame
seq.append(frames[last]); ftype.append(0); ifrac.append(0.0); tt.append(times[last])

ds = xr.Dataset(
    {"brightness_temperature": (("time", "y", "x"), np.stack(seq).astype("float32"),
        {"units": "K", "long_name": "ABI C13 brightness temperature (observed or interpolated)",
         "standard_name": "toa_brightness_temperature"}),
     "frame_type": (("time",), np.array(ftype, dtype="int8"),
        {"long_name": "0 = observed, 1 = SYNTHETIC interpolated", "flag_values": np.array([0, 1], dtype="int8"),
         "flag_meanings": "observed synthetic_interpolated"}),
     "interp_fraction": (("time",), np.array(ifrac, dtype="float32"),
        {"long_name": "temporal fraction between bracketing observed frames"})},
    coords={"time": np.array(tt, dtype="datetime64[ns]")},
    attrs={"title": "FrameFill temporal interpolation of GOES-19 ABI Channel 13",
           "warning": "Frames with frame_type=1 are SYNTHETIC model output, not observations. Do not use for retrievals.",
           "source": "NOAA GOES-19 ABI-L2-CMIPF C13 (public NOAA data, modified here)",
           "method": "50/50 blend of OpenCV DIS flow-warp and RIFE v4.22 (scale 2.0)",
           "bt_range_k": "180-330", "Conventions": "CF-1.8", "roi_pixels": "y=2400:2912, x=2400:2912"})

out = f"{W}/framefill_dayA_10min.nc"
try:
    ds.to_netcdf(out, engine="h5netcdf", encoding={"brightness_temperature": {"zlib": True, "complevel": 4}})
except Exception as e:
    print("compression failed, writing plain:", repr(e)[:80]); ds.to_netcdf(out, engine="h5netcdf")

chk = xr.open_dataset(out, engine="h5netcdf")
print(chk)
n_syn = int((chk.frame_type == 1).sum()); n_obs = int((chk.frame_type == 0).sum())
print(f"frames: {chk.sizes['time']} | observed: {n_obs} | synthetic: {n_syn} | size MB: {round(os.path.getsize(out)/1e6, 1)}")
dt = np.diff(chk.time.values).astype("timedelta64[m]").astype(int)
print("time step (min) unique:", sorted(set(dt)))
print("warning attr:", chk.attrs["warning"])

In [ ]:
import numpy as np, xarray as xr
from PIL import Image, ImageDraw
from matplotlib import cm

W = "/kaggle/working"
LO, HI = 180.0, 330.0
ds = xr.open_dataset(f"{W}/framefill_dayA_10min.nc", engine="h5netcdf")
bt = ds.brightness_temperature.values
ftype = ds.frame_type.values
truth = np.load(f"{W}/crops_dayA.npz")["frames"][:len(bt)]     # nc frame n == observed frame n
times = ds.time.values.astype("datetime64[m]")

def colour(x, size=384):
    n = np.clip((np.nan_to_num(x, nan=LO) - LO) / (HI - LO), 0, 1)
    rgb = (cm.viridis(1 - n)[..., :3] * 255).astype("uint8")     # cold cloud = bright
    return Image.fromarray(rgb).resize((size, size), Image.BILINEAR)

def label(img, text, badge=None):
    d = ImageDraw.Draw(img)
    d.rectangle([0, 0, img.width, 22], fill=(0, 0, 0))
    d.text((6, 5), text, fill=(255, 255, 255))
    if badge:
        d.rectangle([0, img.height - 24, 120, img.height], fill=(200, 60, 0))
        d.text((6, img.height - 18), badge, fill=(255, 255, 255))
    return img

N = 36                                                           # first 6 hours
gif = []
for n in range(N):
    held = bt[2 * (n // 2)]                                      # last real 20-min frame
    p1 = label(colour(held), f"Real 20-min feed  {str(times[n])[11:16]}Z")
    syn = ftype[n] == 1
    p2 = label(colour(bt[n]), "FrameFill 10-min", "SYNTHETIC" if syn else "OBSERVED")
    p3 = label(colour(truth[n]), "Real 10-min (truth)")
    canvas = Image.new("RGB", (384 * 3, 384))
    for k, p in enumerate((p1, p2, p3)):
        canvas.paste(p, (384 * k, 0))
    gif.append(canvas)

out = f"{W}/framefill_compare.gif"
gif[0].save(out, save_all=True, append_images=gif[1:], duration=300, loop=0)
import os; print("saved", out, round(os.path.getsize(out) / 1e6, 1), "MB")
from IPython.display import Image as IPImage, display
display(IPImage(filename=out))